In [1]:
import itertools
from datetime import date
import pathlib
import random
import cv2
import numpy as np
from numba import jit
from IPython import display
import imageio
import ray
from skimage import exposure

In [25]:
# to get the 3D ndarray of per pixel rbgs. 
img = cv2.imread("/Users/derrickvanfrausum/BeCode_AI/git-repos/learn-ai/opencv/assets/images/avatar_dd/db_crypto/ethereum-eth-logo.png")
print(img.shape)

# to flatten the 3D array by one dimension to a 2D array we need.
all_rgb_codes = img.reshape(-1, img.shape[-1])
print(all_rgb_codes.shape)
unique_rgbs = np.unique(all_rgb_codes, axis=0)

unique_rgbs

(3258, 3258, 3)
(10614564, 3)


array([[  0,   0,   0],
       [ 20,  20,  20],
       [ 21,  21,  21],
       [ 22,  22,  22],
       [ 23,  23,  23],
       [ 25,  25,  25],
       [ 26,  26,  26],
       [ 27,  27,  27],
       [ 28,  28,  28],
       [ 29,  29,  29],
       [ 31,  31,  31],
       [ 32,  32,  32],
       [ 33,  33,  33],
       [ 34,  34,  34],
       [ 35,  35,  35],
       [ 36,  36,  36],
       [ 37,  37,  37],
       [ 38,  38,  38],
       [ 39,  39,  39],
       [ 40,  40,  40],
       [ 41,  41,  41],
       [ 42,  42,  42],
       [ 43,  43,  43],
       [ 45,  45,  45],
       [ 46,  46,  46],
       [ 47,  47,  47],
       [ 48,  48,  48],
       [ 49,  49,  49],
       [ 50,  50,  50],
       [ 51,  51,  51],
       [ 51,  68,  68],
       [ 52,  52,  52],
       [ 53,  53,  53],
       [ 54,  54,  54],
       [ 55,  55,  55],
       [ 55,  64,  64],
       [ 56,  56,  56],
       [ 56,  64,  64],
       [ 57,  57,  57],
       [ 57,  61,  61],
       [ 57,  62,  62],
       [ 58,  58

In [26]:
len(unique_rgbs)

122

In [27]:
# Convert it to HSV
img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
img_hsv

array([[[  0,   0, 255],
        [  0,   0, 255],
        [  0,   0, 255],
        ...,
        [  0,   0, 255],
        [  0,   0, 255],
        [  0,   0, 255]],

       [[  0,   0, 255],
        [  0,   0, 255],
        [  0,   0, 255],
        ...,
        [  0,   0, 255],
        [  0,   0, 255],
        [  0,   0, 255]],

       [[  0,   0, 255],
        [  0,   0, 255],
        [  0,   0, 255],
        ...,
        [  0,   0, 255],
        [  0,   0, 255],
        [  0,   0, 255]],

       ...,

       [[  0,   0, 255],
        [  0,   0, 255],
        [  0,   0, 255],
        ...,
        [  0,   0, 255],
        [  0,   0, 255],
        [  0,   0, 255]],

       [[  0,   0, 255],
        [  0,   0, 255],
        [  0,   0, 255],
        ...,
        [  0,   0, 255],
        [  0,   0, 255],
        [  0,   0, 255]],

       [[  0,   0, 255],
        [  0,   0, 255],
        [  0,   0, 255],
        ...,
        [  0,   0, 255],
        [  0,   0, 255],
        [  0,   0, 255]]

In [34]:
# Convert it to HSV
img_hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
np.mean(img_hsv[:,:,1])

0.15447304288711247

In [33]:
np.array(cv2.mean(img_hsv[:,:,1])).astype(np.uint8).tolist()

[0, 0, 0, 0]

In [3]:
img[0,0,:3]

array([15, 13, 19], dtype=uint8)

In [24]:
# set project directory & directory where to save mosaic results images
project_dir = pathlib.Path('/Users/derrickvanfrausum/BeCode_AI/git-repos/learn-ai/opencv/assets/images/putin')
project_name = "putin"
mosaic_images_dir = project_dir / f"mosaic_results_{project_name}"
mosaic_images_dir.mkdir(parents=True, exist_ok=True)

# set text to write
text = "MURDERER"

# set font
font = cv2.FONT_HERSHEY_SIMPLEX

# fontScale
fontScale = 0.5

# Line thickness of 2 px
thickness = 0

# choose number of tasks
nb_tasks = 4

# get photo to recreate
photo_path = "/Users/derrickvanfrausum/BeCode_AI/git-repos/learn-ai/opencv/assets/images/putin/to_recreate/vladimir-putin-illustration-time.jpeg"
print(photo_path)
photo = cv2.imread(photo_path)

# set grid caracteristics
number_rows = 120
number_cols = 120


# read and get first image shape as reference
# ref_img = cv2.imread(img_paths_chunks[0][0])
ref_img = photo
ref_img_height, ref_img_width, ref_img_channel = ref_img.shape
ref_img_height, ref_img_width = ref_img_height//5, ref_img_width//5

# set output grid image shape
out_img_height = ref_img_height * number_rows
out_img_width = ref_img_width * number_cols
out_img_channel = ref_img_channel

photo = cv2.resize(photo, (out_img_width, out_img_height))

# set image matrix canvas
img_matrix = np.zeros((out_img_height, out_img_width, out_img_channel), np.uint8)
# img_matrix.fill(255)

# shutdown ray
ray.shutdown()

# start ray
ray.init(log_to_driver=True)


# @jit(nopython=True)
@ray.remote
def create_mosaic_same_img(photo, text, number_rows, number_cols, grid_range):
# def create_mosaic_same_img(photo, imgs, number_rows, number_cols, grid_range):
    """
    Function to recreate given photo with photos mosaic
    Arguments:
    * grid_range: to assign grid range for each worker
    """

    # create empty dictionary to track grid positions and their corresponding image numpy arrays
    imgs_dict = {}

    # set image matrix canvas
    text_img = np.zeros((ref_img_height, ref_img_width , ref_img_channel), np.uint8)
    # text_img.fill(255)

    # loop through grid cell positions
    grid_positions = itertools.product(range(*grid_range), range(number_rows))
    for (x_i, y_i) in grid_positions:
        print(x_i, y_i)

        # copy canvas for text image
        img = text_img.copy()

        # get region of interest in given photo to recreate
        x = x_i * (ref_img_width)
        y = y_i * (ref_img_height)
        roi = photo[y:y + ref_img_height, x:x + ref_img_width, :]
        
        # get mean bgr values of roi
        # roi_mean = np.array(cv2.mean(roi)[:-1]).astype(np.uint8)
        roi_mean_bgr = np.array(cv2.mean(roi)[:-1]).astype(np.uint8).tolist()

        # get boundary of this text
        text_size = cv2.getTextSize(text, font, fontScale, thickness)[0]

        # get coords based on boundary
        text_x = (img.shape[1] - text_size[0]) // 2
        text_y = (img.shape[0] + text_size[1]) // 2

        # add text
        text_img = cv2.putText(img, text, (text_x, text_y), font, fontScale,
        roi_mean_bgr, thickness, cv2.LINE_AA, False)


        imgs_dict[(y,x)] = img

        # # clear previous output when new fake images are displayed
        # display.clear_output(wait=True)
    
    # # save output image
    # cv2.imwrite(out_img_path, img_matrix)
    
    return imgs_dict


# start tasks in parallel
result_ids = []
# chunk_nb = 0
for i in range(0, number_cols, number_cols//nb_tasks): # set grid range for each worker

    # recreate given photo with photos mosaic
    # result_ids.append(create_mosaic_same_img.remote(photo, img_paths_   chunks[chunk_nb].tolist(), number_rows, number_cols, (i, i+number_cols//nb_tasks)))
    result_ids.append(create_mosaic_same_img.remote(photo, text, number_rows, number_cols, (i, i+number_cols//nb_tasks)))
    # result_ids.append(create_mosaic_same_img.remote(photo, imgs, number_rows, number_cols, (i, i+number_cols//nb_tasks)))

    # # increment chunk number
    # chunk_nb += 1
    
# wait for the tasks to complete and retrieve the results
results = ray.get(result_ids)

# create list to store best match path and xy grid positions
best_match_paths = []
# best_match_indexes = []

# combine results and save image matrix
for result in results:
    for key, cell_image in result.items():
        # best_match_index, best_degree, y, x = key
        y, x = key
        # best_match_index, y, x = key
        img_matrix[y:y+ref_img_height, x:x+ref_img_width, :] = cell_image

        # add result metadata to list
        best_match_paths.append(key)
        # best_match_indexes.append(key)
        # best_match_indexes.append((img_paths[best_match_index], y, x))

# get current date
today = date.today().strftime("%Y%m%d")

# set output image & metadata paths
out_img_path = str(mosaic_images_dir / f"matrix_photo_{number_rows*number_cols}_{today}.jpg")
# out_metadata_path = str(mosaic_images_dir / f"best_match_paths_{number_rows*number_cols}_{today}.npy")
out_metadata_path = str(mosaic_images_dir / f"best_match_paths_{number_rows*number_cols}_{today}.npy")

# save output image & metadata
cv2.imwrite(out_img_path, img_matrix)
np.save(out_metadata_path, np.array(best_match_paths))
# np.save(out_metadata_path, np.array(best_match_indexes))


 # shutdown ray
ray.shutdown()

/Users/derrickvanfrausum/BeCode_AI/git-repos/learn-ai/opencv/assets/images/putin/to_recreate/vladimir-putin-illustration-time.jpeg
(create_mosaic_same_img pid=63852) 0 0
(create_mosaic_same_img pid=63852) 0 1
(create_mosaic_same_img pid=63852) 0 2
(create_mosaic_same_img pid=63852) 0 3
(create_mosaic_same_img pid=63852) 0 4
(create_mosaic_same_img pid=63852) 0 5
(create_mosaic_same_img pid=63852) 0 6
(create_mosaic_same_img pid=63852) 0 7
(create_mosaic_same_img pid=63852) 0 8
(create_mosaic_same_img pid=63852) 0 9
(create_mosaic_same_img pid=63852) 0 10
(create_mosaic_same_img pid=63852) 0 11
(create_mosaic_same_img pid=63852) 0 12
(create_mosaic_same_img pid=63852) 0 13
(create_mosaic_same_img pid=63852) 0 14
(create_mosaic_same_img pid=63852) 0 15
(create_mosaic_same_img pid=63852) 0 16
(create_mosaic_same_img pid=63852) 0 17
(create_mosaic_same_img pid=63852) 0 18
(create_mosaic_same_img pid=63852) 0 19
(create_mosaic_same_img pid=63852) 0 20
(create_mosaic_same_img pid=63852) 0 21

2022-02-24 15:56:42,148	WARNING worker.py:462 -- The driver may not be able to keep up with the stdout/stderr of the workers. To avoid forwarding logs to the driver, use 'ray.init(log_to_driver=False)'.
2022-02-24 15:56:42,195	WARNING worker.py:462 -- The driver may not be able to keep up with the stdout/stderr of the workers. To avoid forwarding logs to the driver, use 'ray.init(log_to_driver=False)'.
2022-02-24 15:56:42,287	WARNING worker.py:462 -- The driver may not be able to keep up with the stdout/stderr of the workers. To avoid forwarding logs to the driver, use 'ray.init(log_to_driver=False)'.


(create_mosaic_same_img pid=63852) 12 38
(create_mosaic_same_img pid=63852) 12 39
(create_mosaic_same_img pid=63852) 12 40
(create_mosaic_same_img pid=63852) 12 41
(create_mosaic_same_img pid=63852) 12 42
(create_mosaic_same_img pid=63852) 12 43
(create_mosaic_same_img pid=63852) 12 44
(create_mosaic_same_img pid=63852) 12 45
(create_mosaic_same_img pid=63852) 12 46
(create_mosaic_same_img pid=63852) 12 47
(create_mosaic_same_img pid=63852) 12 48
(create_mosaic_same_img pid=63852) 12 49
(create_mosaic_same_img pid=63852) 12 50
(create_mosaic_same_img pid=63852) 12 51
(create_mosaic_same_img pid=63852) 12 52
(create_mosaic_same_img pid=63852) 12 53
(create_mosaic_same_img pid=63852) 12 54
(create_mosaic_same_img pid=63852) 12 55
(create_mosaic_same_img pid=63852) 12 56
(create_mosaic_same_img pid=63852) 12 57
(create_mosaic_same_img pid=63852) 12 58
(create_mosaic_same_img pid=63852) 12 59
(create_mosaic_same_img pid=63852) 12 60
(create_mosaic_same_img pid=63852) 12 61
(create_mosaic_s